In [1]:
!pip install supar openai sentencepiece wandb nltk spacy textstat rouge-score bert-score numpy openai language_tool_python==2.9.2 scikit_learn scipy tqdm bitsandbytes transformers Pillow ftfy regex flake8 yapf isort yacs gdown tb-nightly future wilds tabulate

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.7/54.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.3/54.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 22.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 88.1 MB/s eta 0:00:00:00:

In [2]:
import sys
import importlib.util
import os

# Define the parent directory (Plum-main) and utils folder path
plum_main_path = "/kaggle/input/natural-instructions-14/Plum - summarization"
# Add Plum-main to sys.path to allow relative imports inside utils
sys.path.append(plum_main_path)

In [3]:
import os

# Specify the full path where you want to create the directory
path = "/kaggle/working/logs"

# Create the directory
os.makedirs(path, exist_ok=True)

print(f"Directory created at: {path}")

Directory created at: /kaggle/working/logs


In [4]:
#!/usr/bin/env python

import time, gc, os, re, json, random, string, heapq, logging, argparse
import numpy as np
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.tokenize.treebank import TreebankWordDetokenizer
from scipy.stats import entropy
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import difflib
from nltk.metrics.distance import edit_distance
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
# import textstat
import matplotlib.pyplot as plt

# Suppress Warnings
import warnings
warnings.filterwarnings("ignore")
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

from tqdm import tqdm

# External Libraries for Grammar Checking
import spacy
import language_tool_python

# Argument Parsing
parser = argparse.ArgumentParser(description='Multi-objective Genetic Search for Prompt Optimization')
parser.add_argument('--mode', default="Instruction Only", help='Mode of instructions/prompts')
parser.add_argument('--num-shots', default=2, type=int, help='Number of examples in the prompt if applicable')
parser.add_argument('--batch-size', default=1, type=int, help='Batch size')
parser.add_argument('--task-idx', default=1, type=int, help='Index of the task')
parser.add_argument('--seed', default=3, type=int, help='Seed for sampling examples')
parser.add_argument('--train-seed', default=42, type=int, help='Seed for splitting and sampling edit operations')
parser.add_argument('--num-compose', default=1, type=int, help='Number of edits composed for one candidate')
parser.add_argument('--num-train', default=100, type=int, help='Number of examples in score set')
parser.add_argument('--level', default="phrase", help='Level for edit operations')
parser.add_argument('--simulated-anneal', action='store_true', default=False, help='Use simulated anneal when candidate score <= base score')
parser.add_argument('--agnostic', action='store_true', default=False, help='Use template task-agnostic instruction')
parser.add_argument('--print-orig', action='store_true', default=True, help='Print original instruction and evaluate its performance')
parser.add_argument('--write-preds', action='store_true', default=True, help='Store predictions in a JSON file')
parser.add_argument('--meta-dir', default='/kaggle/working/', help='Directory to store metadata')
parser.add_argument('--meta-name', default='modified_language_tool_grammar_qwen2.5_mutation_search.txt', help='Metadata filename')
parser.add_argument('--patience', default=3, type=int, help='Max patience')
parser.add_argument('--num-candidates', default=5, type=int, help='Number of candidates per iteration')
parser.add_argument('--num-iter', default=10, type=int, help='Maximum iterations')
parser.add_argument('--key-id', default=None, type=int, help='OpenAI key ID if applicable')
parser.add_argument('--edits', nargs="+", default=['del', 'swap', 'sub', 'add'], help='Edit operations')
parser.add_argument('--tournament-selection', default=2, type=int, help='Number of tournament selections')
parser.add_argument('--population-size', default=25, type=int, help='Population size for GA')
parser.add_argument('--num-offspring', default=0, type=int, help='Number of offspring per iteration')
parser.add_argument('--mutation-prob', default=0.5, type=float, help='Mutation probability')
parser.add_argument('--data-dir', default='/kaggle/input/natural-instructions-14/Plum - summarization/data/natural-instructions-2.6/tasks/', help='Dataset directory')
parser.add_argument('--project-name', default='NEW_qwen_2.5_25_3patience_good2good_multiple_may_13_summarization_task_1', help='WandB project name')
parser.add_argument('--budget', default=7000, type=int, help='API call budget')
args, unknown = parser.parse_known_args()

# Clear score files at the start of the run
for fname in ['rouge_scores.txt', 'bert_scores.txt', 'bleu_scores.txt']:
    open(os.path.join(args.meta_dir, fname), 'w').close()

tool = language_tool_python.LanguageTool('en-US')

# Initialize progress bar
global_progress_bar = tqdm(total=args.budget, desc="API Calls", leave=False, position=0, dynamic_ncols=True)

try:
    import wandb
    wandb.login(key='5dadd30326caa9de0ea95bb6f1c401782f469e83')
    wandb.init(project=args.project_name)
    wandb.config.update(args)
    tqdm.write("Wandb is setup\n")
except Exception as e:
    tqdm.write("Error while init\n")

# Model Setup (Qwen2.5-7B-Instruct-1M with 4-bit quantization)
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import gc
from accelerate import init_empty_weights, load_checkpoint_and_dispatch

# Set random seed for reproducibility
torch.random.manual_seed(0)

# Model name
model_name = "Qwen/Qwen2.5-7B-Instruct-1M"

# Initialize model with FP16 and multi-GPU support
try:
    # Load model in FP16 with automatic device mapping
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,  # Use FP16 for full precision
        device_map="auto",  # Split across both GPUs
        trust_remote_code=True,
        low_cpu_mem_usage=True  # Reduce CPU memory during loading
    )
except RuntimeError as e:
    if "CUDA out of memory" in str(e):
        print("CUDA out of memory, clearing cache and retrying...")
        torch.cuda.empty_cache()
        gc.collect()
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
            low_cpu_mem_usage=True
        )

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Generation arguments
generation_args = {
    "max_new_tokens": 50,
    "temperature": 0.1,
    "do_sample": False
}

# Verify GPU utilization
print("GPU Utilization:")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.memory_allocated(i) / 1024**3:.2f} GB allocated, "
          f"{torch.cuda.memory_reserved(i) / 1024**3:.2f} GB reserved")

# Initialize Evaluation cache
evaluation_cache = {}

# Instruction Encoding Functions
def encode_instruction(task, instruction_structure=['Definition'], number_of_examples=0, number_of_instances=100, null_word=None, data_seed=0, modified={}, args=args):
    random.seed(0)
    with open(os.path.join(args.data_dir, task)) as json_file:
        data = json.load(json_file)
    instances = data["Instances"]
    instance_indices = list(range(len(instances)))
    test_indices = random.sample(instance_indices, min(number_of_instances, len(instances)))
    
    random.seed(data_seed)
    total_num_examples = number_of_examples if number_of_examples == -1 else number_of_examples
    pos_examples = data["Positive Examples"]
    chosen_examples = random.sample(pos_examples, min(total_num_examples, len(pos_examples))) if number_of_examples > 0 else []
    
    generic_instruction = ''
    for i in instruction_structure:
        if i not in ['Positive Examples Full Only','Positive Examples Full Explanations','Negative Examples Full Explanations']:
            if data[i] != '-':
                if i in modified:
                    data[i] = modified[i]
                data[i] = data[i].replace('\nThings to avoid: -', '').replace('\nEmphasis & Caution: -', '')
                if generic_instruction == '':
                    generic_instruction = i + ': ' + data[i].strip()
                else:
                    generic_instruction += "\n" + i + ': ' + data[i].strip()
        elif i == 'Positive Examples Full Only':
            for j in range(len(chosen_examples)):
                if 'examples' in modified:
                    generic_instruction += "\n" + 'input: ' + modified['examples'][j]['input'] + "\n" + 'output: ' + modified['examples'][j]['output']
                else:
                    generic_instruction += "\n" + 'input: ' + chosen_examples[j]['input'] + "\n" + 'output: ' + chosen_examples[j]['output']
    
    promptlist = []
    answerlist = []
    for i in test_indices:
        if null_word is None:
            if 'input' in modified:
                prompt = generic_instruction + "\n" + instances[i]['input'] + " " + modified['input'] + "\nSummary:"
            else:
                prompt = generic_instruction + "\n" + instances[i]['input'] + "\nSummary:"
        else:
            prompt = generic_instruction + "\n" + null_word + "\nSummary:"
        promptlist.append([{"role": "system", "content": "You are an expert in text summarization."}, {"role": "user", "content": prompt}])
        answerlist.append(instances[i]['output'][0] if isinstance(instances[i]['output'], list) else instances[i]['output'])
    return promptlist, answerlist, test_indices

def training_encode_instruction(task, instruction_structure=['Definition'], number_of_examples=0, number_of_instances=100, null_word=None, data_seed=0, modified={}, args=args):
    random.seed(0)
    with open(os.path.join(args.data_dir, task)) as json_file:
        data = json.load(json_file)
    instances = data["Instances"]
    instance_indices = list(range(len(instances)))
    test_indices = random.sample(instance_indices, min(number_of_instances, len(instances)))
    remaining_indices = [i for i in instance_indices if i not in test_indices]
    
    random.seed(data_seed)
    total_num_examples = number_of_examples if number_of_examples == -1 else number_of_examples
    pos_examples = data["Positive Examples"]
    chosen_examples = random.sample(pos_examples, min(total_num_examples, len(pos_examples))) if number_of_examples > 0 else []
    
    generic_instruction = ''
    for i in instruction_structure:
        if i not in ['Positive Examples Full Only','Positive Examples Full Explanations','Negative Examples Full Explanations']:
            if data[i] != '-':
                if i in modified:
                    data[i] = modified[i]
                data[i] = data[i].replace('\nThings to avoid: -','').replace('\nEmphasis & Caution: -','')
                if generic_instruction == '':
                    generic_instruction = i + ': ' + data[i].strip()
                else:
                    generic_instruction += "\n" + i + ': ' + data[i].strip()
        elif i == 'Positive Examples Full Only':
            for j in range(len(chosen_examples)):
                generic_instruction += "\n" + 'input: ' + chosen_examples[j]['input'] + "\n" + 'output: ' + chosen_examples[j]['output']
    
    promptlist = []
    answerlist = []
    for i in test_indices:
        prompt = generic_instruction + "\n" + instances[i]['input'] + "\nSummary:"
        promptlist.append([{"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."}, {"role": "user", "content": prompt}])
        answerlist.append(instances[i]['output'][0] if isinstance(instances[i]['output'], list) else instances[i]['output'])
    
    train_promptlist, train_answerlist, train_indexlist = [], [], []
    dev_promptlist, dev_answerlist, dev_index_list = [], [], []
    train_indices = random.sample(remaining_indices, min(args.num_train, len(remaining_indices)))
    for i in train_indices:
        prompt = generic_instruction + "\n" + instances[i]['input'] + "\nSummary:"
        train_promptlist.append([{"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."}, {"role": "user", "content": prompt}])
        train_answerlist.append(instances[i]['output'][0] if isinstance(instances[i]['output'], list) else instances[i]['output'])
        train_indexlist.append(i)
    return promptlist, answerlist, test_indices, train_promptlist, train_answerlist, train_indexlist, dev_promptlist, dev_answerlist, dev_index_list

def create_batches(test_instances, test_labels=[], batch_size=4):
    test_sentence_batches = []
    test_label_batches = []
    for i in range(0, len(test_instances), batch_size):
        test_sentence_batches.append(test_instances[i:i+batch_size])
        if len(test_labels) > 0:
            test_label_batches.append(test_labels[i: i + batch_size])
    return (test_sentence_batches, test_label_batches) if test_labels else test_sentence_batches

def construct_instruction_prompt(mode, task_name, num_shots, num_test_instances, data_seed, null_word=None, args=args):
    if mode == "Instruction Only":
        prompt_list, answer_list, index_list = encode_instruction(task_name, instruction_structure=["Definition"], 
                                                                  number_of_examples=num_shots, number_of_instances=num_test_instances, 
                                                                  data_seed=data_seed, null_word=null_word, args=args)
    else:
        raise ValueError("Invalid mode entry, mode not recognized")
    return prompt_list, answer_list, index_list

def counter(func):
    def wrapper(*args, **kwargs):
        wrapper.count += 1
        global_progress_bar.update(1)
        return func(*args, **kwargs)
    wrapper.count = 0
    return wrapper

@counter
def complete_phi4(prompt, max_tokens=None):
    # Qwen2.5 expects a list of messages (system, user)
    messages = prompt  # Prompt is already a list of messages from encode_instruction
    args_local = generation_args.copy()
    if max_tokens:
        args_local["max_new_tokens"] = max_tokens
    
    # Apply chat template and tokenize
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # Generate
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=args_local["max_new_tokens"],
            temperature=args_local["temperature"],
            do_sample=args_local["do_sample"]
        )
    
    # Decode only the generated tokens
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    return response

def run(mode, batch_size, num_shots, chosen_task_name, num_samples, data_seed=0, override_prompts=False, function=None, split=None, modified={}, args=args):
    if not override_prompts:
        prompt_list, answer_list, index_list = construct_instruction_prompt(mode=mode, task_name=chosen_task_name, num_shots=num_shots, num_test_instances=num_samples, data_seed=data_seed, args=args)
    else:
        prompt_list, answer_list, index_list = function(mode=mode, task_name=chosen_task_name, num_shots=num_shots, num_test_instances=num_samples, data_seed=data_seed, null_word=None, split=split, modified=modified, args=args)
    
    prompt_batches, batch_test_labels = create_batches(prompt_list, answer_list, batch_size)
    predictions = []
    
    for batch in prompt_batches:
        for prompt in batch:
            pred = complete_phi4(prompt)
            predictions.append(pred)
    
    rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = [rouge_scorer_obj.score(ref, pred)['rougeL'].fmeasure for ref, pred in zip(answer_list, predictions)]
    bert_scores = bert_score(predictions, answer_list, lang="en", verbose=False)[2].tolist()
    # Compute BLEU scores
    bleu_scores = []
    smoothie = SmoothingFunction().method4
    for pred, ref in zip(predictions, answer_list):
        pred_tokens = word_tokenize(pred.lower())
        ref_tokens = word_tokenize(ref.lower())
        bleu = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothie)
        bleu_scores.append(bleu)
    
    avg_rouge = np.mean(rouge_scores) * 100
    avg_bert = np.mean(bert_scores) * 100
    avg_bleu = np.mean(bleu_scores) * 100
    
    return predictions, avg_rouge, avg_bert, avg_bleu, answer_list, index_list

# Initial Setup
meta_path = os.path.join(args.meta_dir, args.meta_name)
meta_file = open(meta_path, 'w+')
batch_size = args.batch_size
num_shots = args.num_shots
mode = args.mode
seed = args.seed
train_seed = args.train_seed

summarization_task_ids = ['1290', '1357', '1553']
data_files = os.listdir(args.data_dir)
file_map = {f.split("_")[0]: f for f in data_files}
assert args.task_idx >= 0 and args.task_idx < len(summarization_task_ids), "Invalid task index"
chosen_task = summarization_task_ids[args.task_idx]
chosen_task_name = file_map.get('task'+chosen_task, chosen_task)
tqdm.write("Running Experiment for: " + chosen_task_name)

if 'wandb' in globals():
    wandb.log({"Experiment":"Running Experiment for:"+chosen_task_name})

nltk.download('punkt')
file_contents = json.load(open(os.path.join(args.data_dir, chosen_task_name)))
num_samples = 100
num_train_samples = args.num_train

np.random.seed(args.train_seed)
torch.manual_seed(args.train_seed)
population_size = args.population_size
num_compose = args.num_compose
num_candidates = args.num_candidates
num_steps = args.num_iter
num_tournaments = args.tournament_selection
T_max = 10
edit_operations = args.edits
use_add = 'add' in args.edits
num_offspring = args.num_offspring
mutation_prob = args.mutation_prob

if 'wandb' in globals():
    wandb.log({"Num Compose":num_compose,"Num Candidates":num_candidates,"Max Iterations":num_steps,
               "Tournament Selections":num_tournaments,"Edit Operations":edit_operations,
               "Population Size":population_size,"Num Offspring":num_offspring,"Patience":args.patience,
               "Mutation Probability":mutation_prob})

# Grammar Checking Functions
from supar import Parser
parser = Parser.load('crf-con-en')

def get_phrases(instruction):
    phrases = []
    for sentence in sent_tokenize(instruction):
        parsed_tree = parser.predict(word_tokenize(sentence), verbose=False).sentences[0].trees[0]
        leaves = collect_leaves(parsed_tree)
        phrases.extend(leaves)
    normalized_phrases = []
    # List of one-word pronouns to exclude only if they stand alone
    exclude_words = {'he', 'she', 'it', 'they', 'i', 'we', 'me', 'him', 'her', 'them', 'us'}
    for phrase in phrases:
        if phrase not in string.punctuation and phrase != '':
            norm = re.sub(r'\s+', ' ', phrase.strip())
            norm = re.sub(r',\s*', ', ', norm)
            norm = re.sub(r"(')(\S+)(\s+)(')", r"\1\2\4", norm)
            # Skip one-word phrases that are non-significant
            tokens = word_tokenize(norm.lower())
            if len(tokens) == 1 and tokens[0] in exclude_words:
                continue
            normalized_phrases.append(norm)
    return normalized_phrases

def get_phrase_lookup_pun(base_candidate):
    phrases = []
    for sentence in sent_tokenize(base_candidate):
        parsed_tree = parser.predict(word_tokenize(sentence), verbose=False).sentences[0].trees[0]
        leaves = collect_leaves(parsed_tree)
        phrases.extend(leaves)
    phrases = [detokenize(word_tokenize(phrase)) for phrase in phrases if phrase.strip() and not all(c in string.punctuation for c in phrase.strip())]
    phrase_lookup = {p: phrase for p, phrase in enumerate(phrases)}
    tqdm.write(f"Extracted phrases (punctuation excluded): {phrases}")
    meta_file.write(f"Extracted phrases (punctuation excluded): {str(phrases)}\n")
    return phrase_lookup

def collect_leaves(parsed_tree):
    leaves = []
    for tree in parsed_tree:
        if not isinstance(tree, nltk.tree.Tree):
            continue
        if tree.label() == '_': 
            leaves.append(detokenize(tree.leaves()))
            continue
        if check_child(tree):
            leaves.append(detokenize(tree.leaves()))
        else:
            leaves.extend(collect_leaves(tree))
    return leaves

def check_child(tree):
    check = False
    count = 0
    total_count = 0
    for subtree in tree:
        total_count += 1
        if isinstance(subtree, nltk.tree.Tree):
            if subtree.label() == '_':
                count += 1
    if count >= total_count - count:
        check = True
    return check

def detokenize(tokens):
    return TreebankWordDetokenizer().detokenize(tokens)

def containenglish(text):
    return bool(re.search('[a-zA-Z]', text))

def get_llm_grammar_score(text):
    system_prompt = (
    "You are a strict grammar evaluator. Score grammar from 0 to 100. "
    "100 = perfect grammar. 0 = very poor grammar. Return only a number, no text."
    )

    user_prompt = (
    "Evaluate the grammar of this text and return a score from 0 to 100.\n"
    "Text:\n\"\"\"\n" + text + "\n\"\"\""
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    try:
        raw_output = complete_phi4(messages, max_tokens=5)
        tqdm.write(f"Raw grammar output for '{text}': '{raw_output}'")
        # Parse output to extract integer
        match = re.match(r'^\[?(\d+)\]?$', raw_output.strip())
        if match:
            score = int(match.group(1))
        else:
            numbers = re.findall(r'\d+', raw_output)
            if numbers:
                score = int(numbers[0])
            else:
                raise ValueError("No valid number found")
        if 0 <= score <= 100:
            return score
        else:
            tqdm.write(f"Invalid score {score} for '{text}', using LanguageTool fallback")
            return language_tool_fallback(text)
    except (ValueError, TypeError) as e:
        tqdm.write(f"Failed to parse '{raw_output}' for '{text}', using LanguageTool fallback: {str(e)}")
        return language_tool_fallback(text)
    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            tqdm.write("CUDA out of memory during grammar scoring, clearing cache")
            torch.cuda.empty_cache()
            gc.collect()
            return language_tool_fallback(text)
        raise e

def language_tool_fallback(text):
    matches = tool.check(text)
    score = 100
    for match in matches:
        if match.ruleId.startswith('MORFOLOGIK_') or match.ruleId == 'UPPERCASE_SENTENCE_START':
            score -= 5
        elif 'AGREEMENT' in match.ruleId:
            score -= 15
        elif 'GRAMMAR' in match.ruleId or 'SENTENCE' in match.ruleId:
            score -= 20
        else:
            score -= 10
    return max(0, score)

def substitute_phrase(candidate, phrase):
    system_prompt = (
    "You are a grammar and paraphrasing expert. Your task is to paraphrase a phrase so it fits grammatically and contextually within an instruction."
    )

    user_prompt = (
        "Given a text and a phrase, provide the best paraphrase for that phrase which fits perfectly in the text.\n"
        "Instructional text: "+ candidate + "\n"
        "Phrase to paraphrase: "+ phrase + "\n"
        "Only return the paraphrased phrase—no extra text or explanation.\n"
        "Paraphrased phrase:"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    try:
        paraphrase = complete_phi4(messages, max_tokens=30).strip('.')
        paraphrase = paraphrase.strip('\'"')
        paraphrase = re.sub(r'^(Paraphrased phrase:)\s*', '', paraphrase)
        if paraphrase.strip() == '':
            tqdm.write("Empty paraphrase generated, returning original phrase")
            paraphrase = phrase
        tqdm.write("Substituting phrase: " + phrase + " with: " + paraphrase)
        # Verify the full prompt after substitution
        if candidate.find(' ' + phrase + ' ') > 0:
            full_prompt = candidate.replace(' ' + phrase + ' ', ' ' + paraphrase + ' ')
        elif candidate.find(phrase + ' ') > 0:
            full_prompt = candidate.replace(phrase + ' ', paraphrase + ' ')
        elif candidate.find(' ' + phrase) > 0:
            full_prompt = candidate.replace(' ' + phrase, ' ' + paraphrase)
        else:
            full_prompt = candidate.replace(phrase, paraphrase)
        grammar_score = get_llm_grammar_score(full_prompt)
        if grammar_score < 10:
            tqdm.write(f"Substituted prompt '{full_prompt}' has low grammar score ({grammar_score}), returning original phrase")
            paraphrase = phrase
    except Exception as e:
        tqdm.write(f"Error during paraphrasing: {e}, returning original phrase")
        paraphrase = phrase
    
    if candidate.find(' ' + phrase + ' ') > 0:
        answer = candidate.replace(' ' + phrase + ' ', ' ' + paraphrase + ' ')
    elif candidate.find(phrase + ' ') > 0:
        answer = candidate.replace(phrase + ' ', paraphrase + ' ')
    elif candidate.find(' ' + phrase) > 0:
        answer = candidate.replace(' ' + phrase, ' ' + paraphrase)
    else:
        answer = candidate.replace(phrase, paraphrase)
    return answer

def delete_phrase(candidate, phrase):
    if candidate.find(' ' + phrase) > 0:
        answer = candidate.replace(' ' + phrase, ' ')
    elif candidate.find(phrase + ' ') > 0:
        answer = candidate.replace(phrase + ' ', ' ')
    else:
        answer = candidate.replace(phrase, '')
    return answer

def add_phrase(candidate, phrase, after):
    if after == '':
        answer = phrase + ' ' + candidate
    else:
        if candidate.find(' ' + after) > 0:
            answer = candidate.replace(' ' + after, ' ' + after + ' ' + phrase)
        elif candidate.find(after + ' ') > 0:
            answer = candidate.replace(after + ' ', after + ' ' + phrase + ' ')
        else:
            answer = candidate.replace(after, after + phrase)
    return answer

def swap_phrases(candidate, phrase_1, phrase_2):
    if phrase_1 in phrase_2:
        if candidate.find(' ' + phrase_2 + ' ') >= 0:
            candidate = candidate.replace(' ' + phrase_2 + ' ', ' <2> ')
        else:
            candidate = candidate.replace(phrase_2, '<2>')
        answer = candidate
        if candidate.find(' ' + phrase_1 + ' ') >= 0:
            answer = answer.replace(' ' + phrase_1 + ' ', ' <1> ')
        else:
            answer = answer.replace(phrase_1, '<1>')
        answer = answer.replace('<1>', phrase_2)
        answer = answer.replace('<2>', phrase_1)
    else:
        if candidate.find(' ' + phrase_1 + ' ') >= 0:
            candidate = candidate.replace(' ' + phrase_1 + ' ', ' <1> ')
        else:
            candidate = candidate.replace(phrase_1, '<1>')
        answer = candidate
        if candidate.find(' ' + phrase_2 + ' ') >= 0:
            answer = answer.replace(' ' + phrase_2 + ' ', ' <2> ')
        else:
            answer = answer.replace(phrase_2, '<2>')
        answer = answer.replace('<1>', phrase_2)
        answer = answer.replace('<2>', phrase_1)
    return answer

def perform_edit(edit, base_text, phrase_list, deleted_phrases_history):
    mutated = base_text
    if edit == 'del':
        if not phrase_list:
            tqdm.write("No matching phrase found for deletion.")
            return base_text, deleted_phrases_history
        chosen = random.choice(phrase_list)
        phrase_list.remove(chosen)
        tqdm.write("Deleting phrase: " + chosen)
        mutated = delete_phrase(base_text, chosen)
        deleted_phrases_history.append(chosen)
    elif edit == 'swap':
        if len(phrase_list) < 2:
            tqdm.write("Not enough matching phrases found for swap.")
            return base_text, deleted_phrases_history
        p1, p2 = random.sample(phrase_list, 2)
        for p in (p1, p2):
            if p in phrase_list:
                phrase_list.remove(p)
        tqdm.write("Swapping phrases: " + p1 + " and " + p2)
        mutated = swap_phrases(base_text, p1, p2)
    elif edit == 'sub':
        if not phrase_list:
            tqdm.write("No matching phrase found for substitution.")
            return base_text, deleted_phrases_history
        chosen = random.choice(phrase_list)
        phrase_list.remove(chosen)
        tqdm.write("Substituting phrase: " + chosen)
        mutated = substitute_phrase(base_text, chosen)
    elif edit == 'add':
        if not deleted_phrases_history:
            tqdm.write("No deleted phrases available for addition.")
            return base_text, deleted_phrases_history
        phrase_to_add = random.choice(deleted_phrases_history)
        if phrase_list:
            after = random.choice(phrase_list)
            if after in phrase_list:
                phrase_list.remove(after)
            tqdm.write("Adding phrase: " + phrase_to_add + " after " + after)
            mutated = add_phrase(base_text, phrase_to_add, after)
        else:
            tqdm.write("No matching phrase found for 'add' operation; prepending phrase: " + phrase_to_add)
            mutated = phrase_to_add + " " + base_text
    return mutated, deleted_phrases_history

def safe_mutation(edit, base_text, phrase_list, deleted_phrases_history, grammar_threshold=50, similarity_threshold=0.0):
    mutated, new_del = perform_edit(edit, base_text, phrase_list, deleted_phrases_history)
    gscore = get_llm_grammar_score(mutated)
    if gscore >= grammar_threshold:
        summarization_score, _, _, _, _ = evaluate_objectives(mutated)
        tqdm.write(f"After applying {edit} operation: grammar score = {gscore}, summarization score = {summarization_score}")
    else:
        tqdm.write(f"After applying {edit} operation: grammar score = {gscore}")
        tqdm.write("Mutation rejected due to low grammar.")
        return base_text, deleted_phrases_history
    return mutated, new_del

def evaluate_objectives(candidate, split='train'):
    # Check if the candidate has already been evaluated
    if candidate in evaluation_cache:
        return evaluation_cache[candidate]
    
    predictions, avg_rouge, avg_bert, avg_bleu, answer_list, indexlist = run(
        mode=args.mode,
        batch_size=args.batch_size,
        num_shots=args.num_shots,
        chosen_task_name=chosen_task_name,
        num_samples=num_samples,
        data_seed=args.seed,
        override_prompts=True,
        function=custom_instruction_prompt,
        split=split,
        modified={'Definition': candidate},
        args=args
    )
    
    summarization_score = 0.6 * avg_rouge + 0.4 * avg_bert
    grammar_score = get_llm_grammar_score(candidate)
    
    # Save scores to separate text files
    with open(os.path.join(args.meta_dir, 'rouge_scores.txt'), 'a') as f_rouge:
        f_rouge.write(f"Candidate: {candidate}\nAverage ROUGE Score: {avg_rouge:.4f}\n\n")
    with open(os.path.join(args.meta_dir, 'bert_scores.txt'), 'a') as f_bert:
        f_bert.write(f"Candidate: {candidate}\nAverage BERT Score: {avg_bert:.4f}\n\n")
    with open(os.path.join(args.meta_dir, 'bleu_scores.txt'), 'a') as f_bleu:
        f_bleu.write(f"Candidate: {candidate}\nAverage BLEU Score: {avg_bleu:.4f}\n\n")
    
    # Save the computed scores in the cache
    evaluation_cache[candidate] = (summarization_score, grammar_score, avg_rouge, avg_bert, avg_bleu)
    return summarization_score, grammar_score, avg_rouge, avg_bert, avg_bleu

def score(candidate, split='test', write=False):
    predictions, avg_rouge, avg_bert, avg_bleu, answer_list, indexlist = run(
        mode=args.mode, batch_size=batch_size, num_shots=num_shots, chosen_task_name=chosen_task_name,
        num_samples=num_samples, data_seed=args.seed, override_prompts=True, function=custom_instruction_prompt,
        split=split, modified={'Definition': candidate}, args=args)
    summarization_score = 0.6 * avg_rouge + 0.4 * avg_bert
    if split == 'test' and write:
        pname = args.meta_name.split('.')[0] + "_predictions.json"
        pred_dump = {'predictions': predictions, 'answers': answer_list, 'ids': indexlist}
        ppath = os.path.join(args.meta_dir, pname)
        with open(ppath, 'w+') as pfile:
            json.dump(pred_dump, pfile)
    return summarization_score

def custom_instruction_prompt(mode=args.mode, task_name=None, num_shots=args.num_shots,
                              num_test_instances=100, data_seed=None, null_word=None, split='train',
                              modified={}, args=args):
    if task_name is None:
        task_name = chosen_task_name
    if mode == "Instruction Only":
        result = training_encode_instruction(
            task_name, instruction_structure=["Definition"],
            number_of_examples=num_shots, number_of_instances=num_test_instances,
            data_seed=data_seed, null_word=null_word, modified=modified, args=args
        )
    else:
        raise ValueError()
    if split == 'test':
        prompt_list, answer_list, index_list = result[:3]
        return prompt_list, answer_list, index_list
    elif split == 'train':
        (prompt_list, answer_list, index_list,
         train_prompt_list, train_answer_list, train_index_list,
         dev_prompt_list, dev_answerlist, dev_index_list) = result
        train_prompt_list.extend(dev_prompt_list)
        train_answer_list.extend(dev_answerlist)
        train_index_list.extend(dev_index_list)
        try:
            random.seed(data_seed)
            indices = random.sample(range(len(train_index_list)), args.num_train)
            train_prompt_list = [train_prompt_list[i] for i in indices]
            train_answer_list = [train_answer_list[i] for i in indices]
            train_index_list = [train_index_list[i] for i in indices]
        except Exception as e:
            tqdm.write(f"Error in sampling: {e}")
        return train_prompt_list, train_answer_list, train_index_list
    else:
        raise ValueError()

def tournament_selection(population, population_scores, num_tournaments):
    S_candidates = []
    S_scores = []
    for _ in range(num_tournaments):
        parent = np.random.randint(0, len(population))
        S_candidates.append(population[parent])
        S_scores.append(population_scores[parent])
    base_idx = np.argmax(S_scores)
    return S_candidates[base_idx], S_scores[base_idx]

def crossover(parent_1, parent_2):
    flag_error = False
    max_attempts = 3
    attempt = 0
    
    while attempt < max_attempts:
        try:
            phrases_1_pun = get_phrase_lookup_pun(parent_1)
            phrases_2_pun = get_phrase_lookup_pun(parent_2)
        except AttributeError:
            tqdm.write("AttributeError during phrase extraction in crossover")
            meta_file.write("AttributeError during phrase extraction in crossover\n")
            if 'wandb' in globals():
                wandb.log({"Crossover Error": "AttributeError during phrase extraction"})
            return parent_1, True
        
        phrases_1 = list(phrases_1_pun.values())
        phrases_2 = list(phrases_2_pun.values())
        
        if not phrases_1 or not phrases_2:
            tqdm.write("No valid phrases for crossover")
            meta_file.write("No valid phrases for crossover\n")
            return parent_1, True
        
        offspring_phrases = []
        total_phrases = min(len(phrases_1), len(phrases_2))
        num_from_p1 = random.randint(1, total_phrases - 1) if total_phrases > 1 else 1
        num_from_p2 = total_phrases - num_from_p1
        
        random.shuffle(phrases_1)
        random.shuffle(phrases_2)
        offspring_phrases.extend(phrases_1[:num_from_p1])
        offspring_phrases.extend(phrases_2[:num_from_p2])
        
        offspring_words = []
        for phrase in offspring_phrases:
            offspring_words.extend(word_tokenize(phrase))
        offspring = detokenize(offspring_words)
        
        grammar_score = get_llm_grammar_score(offspring)
        if containenglish(offspring) and grammar_score >= 50:
            tqdm.write(f"Generated offspring (attempt {attempt + 1}): {offspring}, Grammar: {grammar_score}")
            meta_file.write(f"Generated offspring (attempt {attempt + 1}): {offspring}, Grammar: {grammar_score}\n")
            if 'wandb' in globals():
                wandb.log({"Crossover Offspring": offspring, "Grammar Score": grammar_score, "Attempt": attempt + 1})
            return offspring, flag_error
        else:
            tqdm.write(f"Offspring rejected (attempt {attempt + 1}): Grammar={grammar_score}, English={containenglish(offspring)}")
            meta_file.write(f"Offspring rejected (attempt {attempt + 1}): Grammar={grammar_score}, English={containenglish(offspring)}\n")
            attempt += 1
    
    tqdm.write("All crossover attempts failed, returning parent_1")
    meta_file.write("All crossover attempts failed, returning parent_1\n")
    if 'wandb' in globals():
        wandb.log({"Crossover Failed": "All attempts exhausted"})
    return parent_1, True

def plot_pareto_front(population_data, fronts, iteration, meta_dir, final=False):
    plt.figure(figsize=(8, 6))
    summarization_scores = [data[1] for data in population_data]
    grammar_scores = [data[2] for data in population_data]
    plt.scatter(summarization_scores, grammar_scores, c='gray', alpha=0.5, label='All Candidates')
    
    # Define colors for different fronts
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    for front_idx, front in enumerate(fronts):
        if front_idx >= len(colors):
            break  # Limit to available colors
        front_summ = [population_data[i][1] for i in front]
        front_gram = [population_data[i][2] for i in front]
        sorted_indices = np.argsort(front_summ)
        front_summ_sorted = [front_summ[i] for i in sorted_indices]
        front_gram_sorted = [front_gram[i] for i in sorted_indices]
        label = f'Front {front_idx + 1}' if front_idx > 0 else 'Pareto Front'
        plt.scatter(front_summ, front_gram, c=colors[front_idx], label=label)
        plt.plot(front_summ_sorted, front_gram_sorted, c=colors[front_idx], linestyle='--')
    
    plt.xlabel('Summarization Score')
    plt.ylabel('Grammar Score')
    title = f'Pareto Optimal Curve (Final)' if final else f'Pareto Optimal Curve (Iteration {iteration})'
    plt.title(title)
    plt.legend()
    plt.grid(True)
    
    plot_name = 'pareto_final.png' if final else f'pareto_iteration_{iteration}.png'
    plot_path = os.path.join(meta_dir, plot_name)
    plt.savefig(plot_path)
    plt.close()
    
    if 'wandb' in globals():
        wandb.log({title: wandb.Image(plot_path)})
    return plot_path

# Define multiple initial candidates
# initial_candidates = [
#     "In this task, you are given an article. Your task is to summarize in a sentence.",
#     "You will receive an article in this task. Your goal is to condense it into one sentence.",
#     "This task provides you with an article to read. Summarize it in a single sentence.",
#     "Given an article here, your job is to write a one-sentence summary.",
#     "In this exercise, an article is presented to you. Create a sentence summarizing it.",
#     "You are handed an article for this task. Reduce it to a single summary sentence.",
#     "The task involves an article. Your duty is to summarize it in one sentence.",
#     "An article is supplied in this task. Compose a one-sentence summary of it.",
#     "For this task, you’ll get an article. Express its main idea in a single sentence.",
#     "Here, you are given a piece of writing. Your task is to sum it up in one sentence.",
#     "In this activity, you receive an article. Your objective is to summarize it in one sentence.",
#     "You are provided with an article in this task. Write a single sentence capturing its essence.",
#     "This task gives you an article. Your role is to condense it into a single sentence.",
#     "An article is presented in this task. Summarize its content in one sentence.",
#     "For this exercise, you are given an article. Produce a one-sentence summary.",
#     "In this task, an article is provided. Your goal is to create a one-sentence summary.",
#     "You will be given an article here. Summarize it in a single sentence.",
#     "This task includes an article. Your job is to write one sentence summarizing it.",
#     "An article is given to you in this task. Condense its main point into one sentence.",
#     "In this exercise, you’ll receive an article. Summarize it in a single sentence.",
#     "You are assigned an article for this task. Write a one-sentence summary of it.",
#     "This task involves receiving an article. Your aim is to summarize it in one sentence.",
#     "An article is provided here. Your task is to express its summary in one sentence.",
#     "In this task, you are given a written piece. Summarize it in a single sentence.",
#     "You are presented with an article in this task. Create a one-sentence overview."
# ]

initial_candidates = [
    "Generate an appropriate single-sentence summary for the given text. Ensure that the summary includes the main topic of the text.",
    "Given a text, create a single-sentence summary that captures its main topic.",
    "Your task is to read the provided text and summarize it in one sentence, including the main topic.",
    "For the given text, write a one-sentence summary that reflects its primary topic.",
    "You are provided with a text; your goal is to condense it into a single sentence highlighting the main topic.",
    "In this task, you receive a text and must summarize it in one sentence, focusing on the main topic.",
    "The provided text needs a single-sentence summary that includes its central topic.",
    "Read the given text and produce a one-sentence summary that conveys its main topic.",
    "Your objective is to summarize the provided text in a single sentence, ensuring the main topic is included.",
    "For this exercise, you are given a text to summarize in one sentence, capturing its main topic.",
    "You are tasked with writing a one-sentence summary of the given text, emphasizing its primary topic.",
    "Given a piece of text, your job is to create a single-sentence summary that addresses its main topic.",
    "In this activity, summarize the provided text in one sentence, incorporating its main topic.",
    "You are presented with a text; write a single sentence that summarizes it, including the main topic.",
    "The task is to generate a one-sentence summary for the given text, reflecting its core topic.",
    "For the text provided, compose a single-sentence summary that highlights its main topic.",
    "Your role is to condense the given text into one sentence that captures its primary topic.",
    "In this task, you are given a text to summarize in a single sentence, focusing on its main topic.",
    "Read the provided text and write a one-sentence summary that includes its central topic.",
    "You are given a text; your task is to summarize it in one sentence, ensuring the main topic is clear.",
    "For this task, create a single-sentence summary of the provided text, emphasizing its main topic.",
    "The given text must be summarized in one sentence that conveys its primary topic.",
    "Your goal is to produce a one-sentence summary of the provided text, incorporating its main topic.",
    "In this exercise, you are tasked with summarizing the given text in one sentence, including its core topic.",
    "You are provided with a text and must write a one-sentence summary that captures its main topic."
]

# Ensure population_size matches the number of initial candidates or select top candidates
if args.population_size < len(initial_candidates):
    tqdm.write(f"Population size ({args.population_size}) is less than number of initial candidates ({len(initial_candidates)}). Selecting top {args.population_size} candidates.")
    meta_file.write(f"Population size ({args.population_size}) is less than number of initial candidates ({len(initial_candidates)}). Selecting top {args.population_size} candidates.\n")
elif args.population_size > len(initial_candidates):
    tqdm.write(f"Warning: Population size ({args.population_size}) is greater than number of initial candidates ({len(initial_candidates)}). Duplicating candidates to fill population.")
    meta_file.write(f"Warning: Population size ({args.population_size}) is greater than number of initial candidates ({len(initial_candidates)}). Duplicating candidates to fill population.\n")
    initial_candidates = (initial_candidates * (args.population_size // len(initial_candidates) + 1))[:args.population_size]

# Evaluate each initial candidate
initial_scores = []
for candidate in initial_candidates:
    summarization_score, grammar_score, avg_rouge, avg_bert, avg_bleu = evaluate_objectives(candidate)
    weighted_score = 0.6 * summarization_score + 0.4 * grammar_score
    initial_scores.append({
        "candidate": candidate,
        "summarization_score": summarization_score,
        "grammar_score": grammar_score,
        "rouge_score": avg_rouge,
        "bert_score": avg_bert,
        "bleu_score": avg_bleu,
        "weighted_score": weighted_score
    })
    tqdm.write(f"Initial Candidate: {candidate}")
    tqdm.write(f"Scores (Summarization, Grammar, ROUGE, BERT, BLEU, Weighted): "
              f"({summarization_score:.4f}, {grammar_score:.4f}, {avg_rouge:.4f}, {avg_bert:.4f}, {avg_bleu:.4f}, {weighted_score:.4f})")
    meta_file.write(f"Initial Candidate: {candidate}\n")
    meta_file.write(f"Scores (Summarization, Grammar, ROUGE, BERT, BLEU, Weighted): "
                    f"({summarization_score:.4f}, {grammar_score:.4f}, {avg_rouge:.4f}, {avg_bert:.4f}, {avg_bleu:.4f}, {weighted_score:.4f})\n")
    if 'wandb' in globals():
        wandb.log({
            "Initial Candidate": candidate,
            "Initial Summarization Score": summarization_score,
            "Initial Grammar Score": grammar_score,
            "Initial ROUGE Score": avg_rouge,
            "Initial BERT Score": avg_bert,
            "Initial BLEU Score": avg_bleu,
            "Initial Weighted Score": weighted_score
        })

# Select the best initial candidate based on weighted score
best_initial = max(initial_scores, key=lambda x: x["weighted_score"])
best_candidate = best_initial["candidate"]
best_summarization_score = best_initial["summarization_score"]
best_grammar_score = best_initial["grammar_score"]
best_rouge_score = best_initial["rouge_score"]
best_bert_score = best_initial["bert_score"]
best_bleu_score = best_initial["bleu_score"]

tqdm.write(f"Best Initial Candidate: {best_candidate}")
tqdm.write(f"Best Initial Scores (Summarization, Grammar, ROUGE, BERT, BLEU): "
          f"({best_summarization_score:.4f}, {best_grammar_score:.4f}, {best_rouge_score:.4f}, {best_bert_score:.4f}, {best_bleu_score:.4f})")
meta_file.write(f"Best Initial Candidate: {best_candidate}\n")
meta_file.write(f"Best Initial Scores (Summarization, Grammar, ROUGE, BERT, BLEU): "
                f"({best_summarization_score:.4f}, {best_grammar_score:.4f}, {best_rouge_score:.4f}, {best_bert_score:.4f}, {best_bleu_score:.4f})\n")
if 'wandb' in globals():
    wandb.log({
        "Best Initial Candidate": best_candidate,
        "Best Initial Summarization Score": best_summarization_score,
        "Best Initial Grammar Score": best_grammar_score,
        "Best Initial ROUGE Score": best_rouge_score,
        "Best Initial BERT Score": best_bert_score,
        "Best Initial BLEU Score": best_bleu_score
    })

# Initialize tracking lists with the best initial candidate's scores (iteration 0)
best_rouge_scores = [best_rouge_score]
best_bert_scores = [best_bert_score]
best_bleu_scores = [best_bleu_score]
best_summarization_scores = [best_summarization_score]
best_grammar_scores = [best_grammar_score]

# Initialize population
if len(initial_candidates) <= args.population_size:
    W_candidates = initial_candidates
    W_objectives = [(s["summarization_score"], s["grammar_score"]) for s in initial_scores]
    W_deletesets = [[] for _ in range(len(W_candidates))]
else:
    sorted_scores = sorted(initial_scores, key=lambda x: x["weighted_score"], reverse=True)[:args.population_size]
    W_candidates = [s["candidate"] for s in sorted_scores]
    W_objectives = [(s["summarization_score"], s["grammar_score"]) for s in sorted_scores]
    W_deletesets = [[] for _ in range(len(W_candidates))]
    tqdm.write(f"Selected top {args.population_size} initial candidates based on weighted score.")
    meta_file.write(f"Selected top {args.population_size} initial candidates based on weighted score.\n")

# Log initial population
tqdm.write("Initial Population:")
for candidate, obj in zip(W_candidates, W_objectives):
    info = {"prompt": candidate, "summarization_score": obj[0], "grammar_score": obj[1]}
    tqdm.write(str(info))
if 'wandb' in globals():
    wandb.log({"Initial Population": W_objectives})

# Initialize best candidate for patience logic
plot_pareto_front.best_candidate = best_candidate
plot_pareto_front.patience_counter = 0


WEIGHT_SUMM = 0.6
WEIGHT_GRAM = 0.4

# Main Evolutionary Loop
start_time = time.time()

for iter_idx in range(num_steps):

    tqdm.write("\n----- Iteration: " + str(iter_idx) + " -----")
    meta_file.write("Running step:\t " + str(iter_idx) + '\n')
    
    tqdm.write("Current Population:")
    for candidate, obj in zip(W_candidates, W_objectives):
        info = {"prompt": candidate, "summarization_score": obj[0], "grammar_score": obj[1]}
        tqdm.write(str(info))
    if 'wandb' in globals():
        wandb.log({f"Current Population (start of iteration {iter_idx})": W_objectives})
    
    new_candidates = W_candidates.copy()
    new_objectives = W_objectives.copy()
    new_deletesets = W_deletesets.copy()
    
    for j in range(num_offspring):
        parent_1, _ = tournament_selection(W_candidates, [obj[0] for obj in W_objectives], num_tournaments)
        parent_2, _ = tournament_selection(W_candidates, [obj[0] for obj in W_objectives], num_tournaments)
        meta_file.write("Parent 1 (" + str(j) + "):\t " + parent_1 + '\n')
        meta_file.write("Parent 2 (" + str(j) + "):\t " + parent_2 + '\n')
        tqdm.write("Parent 1 (" + str(j) + "): " + parent_1)
        tqdm.write("Parent 2 (" + str(j) + "): " + parent_2)
        offspring, flag_error = crossover(parent_1, parent_2)
        if flag_error or not containenglish(offspring):
            meta_file.write("Crossover skipped due to error or non-English offspring\n")
            tqdm.write("Crossover skipped due to error or non-English offspring")
            if 'wandb' in globals():
                wandb.log({"Crossover Skipped": f"Iteration {iter_idx}, Offspring {j}"})
            continue
        meta_file.write("Offspring (" + str(j) + "):\t " + offspring + '\n')
        tqdm.write("Offspring (" + str(j) + "): " + offspring)
        new_candidates.append(offspring)
        new_deletesets.append(new_deletesets[W_candidates.index(parent_1)])
    
    for idx, base_candidate in enumerate(new_candidates[:population_size]):
        if mutation_prob > np.random.random():
            try:
                phrase_list = get_phrases(base_candidate)
                tqdm.write("Initial phrases for candidate mutation: " + str(phrase_list))
            except AttributeError:
                tqdm.write("Mutation skipped due to error")
                continue
            if use_add and not new_deletesets[idx]:
                if 'add' in edit_operations:
                    edit_operations.remove('add')
            edits = np.random.choice(edit_operations, num_candidates)
            tqdm.write("Sampled edit operations for mutation: " + str(edits))
            base_grammar_score = W_objectives[idx][1]
            grammar_threshold = 35
            similarity_threshold = 0.0
            for edit_op in edits:
                mutated, new_deletesets[idx] = safe_mutation(
                    edit_op, base_candidate, phrase_list.copy(), new_deletesets[idx],
                    grammar_threshold=grammar_threshold, similarity_threshold=similarity_threshold
                )
                if mutated != base_candidate:
                    new_candidates[idx] = mutated
                    break
    
    new_objectives = []
    candidate_scores = []  # Store all scores for candidates in this iteration
    for candidate in new_candidates:
        summarization_score, grammar_score, avg_rouge, avg_bert, avg_bleu = evaluate_objectives(candidate)
        new_objectives.append((summarization_score, grammar_score))
        candidate_scores.append((avg_rouge, avg_bert, avg_bleu, summarization_score, grammar_score))
    
    combined = list(zip(new_candidates, new_objectives, new_deletesets))
    population_data = [(c, o[0], o[1], d) for c, o, d in combined]
    
    def non_dominated_sorting(population):
        n = len(population)
        domination_count = [0] * n
        dominated_set = [[] for _ in range(n)]
        fronts = []
        
        # Step 1: Compute domination counts and sets
        for i in range(n):
            for j in range(n):
                if i == j:
                    continue
                p_summ, p_gram = population[i][1], population[i][2]
                q_summ, q_gram = population[j][1], population[j][2]
                if (p_summ >= q_summ and p_gram >= q_gram) and (p_summ > q_summ or p_gram > q_gram):
                    dominated_set[i].append(j)
                elif (q_summ >= p_summ and q_gram >= p_gram) and (q_summ > p_summ or q_gram > p_gram):
                    domination_count[i] += 1
        
        # Step 2: Assign first front
        front0 = [i for i in range(n) if domination_count[i] == 0]
        fronts.append(front0)
        
        # Step 3: Construct subsequent fronts
        current_front = front0
        while current_front:
            next_front = []
            for i in current_front:
                for j in dominated_set[i]:
                    domination_count[j] -= 1
                    if domination_count[j] == 0:
                        next_front.append(j)
            if next_front:
                fronts.append(next_front)
            current_front = next_front
        
        return fronts

    def compute_crowding_distance(population_data, front):
        """
        Compute the crowding distance for each individual in the front.
        population_data is a list of tuples: (candidate, summarization_score, grammar_score, deleted_set)
        front is a list of indices (into population_data) that are in one Pareto front.
        """
        # Initialize distances to zero for all individuals in the front.
        distances = {i: 0.0 for i in front}
        num_objectives = 2  # summarization and grammar
        
        # Process each objective individually
        # Objective 1: Summarization Score
        sorted_by_summ = sorted(front, key=lambda i: population_data[i][1])
        summ_min = population_data[sorted_by_summ[0]][1]
        summ_max = population_data[sorted_by_summ[-1]][1]
        distances[sorted_by_summ[0]] = float('inf')
        distances[sorted_by_summ[-1]] = float('inf')
        for k in range(1, len(sorted_by_summ) - 1):
            if summ_max - summ_min == 0:
                norm_diff = 0.0
            else:
                norm_diff = (population_data[sorted_by_summ[k + 1]][1] - population_data[sorted_by_summ[k - 1]][1]) / (summ_max - summ_min)
            distances[sorted_by_summ[k]] += norm_diff

        # Objective 2: Grammar Score
        sorted_by_gram = sorted(front, key=lambda i: population_data[i][2])
        gram_min = population_data[sorted_by_gram[0]][2]
        gram_max = population_data[sorted_by_gram[-1]][2]
        distances[sorted_by_gram[0]] = float('inf')
        distances[sorted_by_gram[-1]] = float('inf')
        for k in range(1, len(sorted_by_gram) - 1):
            if gram_max - gram_min == 0:
                norm_diff = 0.0
            else:
                norm_diff = (population_data[sorted_by_gram[k + 1]][2] - population_data[sorted_by_gram[k - 1]][2]) / (gram_max - gram_min)
            distances[sorted_by_gram[k]] += norm_diff

        return distances

    fronts = non_dominated_sorting(population_data)
    tqdm.write(f"Non-dominated fronts (by candidate indices): {fronts}")
    meta_file.write(f"Non-dominated fronts (by candidate indices): {str(fronts)}\n")
    
    plot_pareto_front(population_data, fronts, iter_idx, args.meta_dir)
    
    final_population_indices = []
    remaining = population_size
    for front in fronts:
        if len(front) <= remaining:
            final_population_indices.extend(front)
            remaining -= len(front)
        else:
            distances = compute_crowding_distance(population_data, front)
            sorted_front = sorted(front, key=lambda i: distances[i], reverse=True)
            final_population_indices.extend(sorted_front[:remaining])
            remaining = 0
        if remaining == 0:
            break

    W_candidates = [population_data[i][0] for i in final_population_indices]
    W_objectives = [(population_data[i][1], population_data[i][2]) for i in final_population_indices]
    W_deletesets = [population_data[i][3] for i in final_population_indices]
    
    tqdm.write("Updated Population at end of iteration {}:".format(iter_idx))
    for candidate, obj in zip(W_candidates, W_objectives):
        info = {"prompt": candidate, "summarization_score": obj[0], "grammar_score": obj[1]}
        tqdm.write(str(info))
    
    # Modified: Select best candidate from Pareto front
    pareto_front = fronts[0]  # First front is Pareto-optimal
    if len(pareto_front) == 1:
        best_idx = pareto_front[0]
    else:
        # Select candidate with highest weighted score (0.6 * summarization + 0.4 * grammar)
        best_idx = max(
            pareto_front,
            key=lambda i: WEIGHT_SUMM * population_data[i][1] + WEIGHT_GRAM * population_data[i][2]
        )
    
    result_candidate = population_data[best_idx][0]
    result_objectives = (population_data[best_idx][1], population_data[best_idx][2])
    
    # Get all scores for the best candidate
    best_scores = candidate_scores[best_idx]  # (avg_rouge, avg_bert, avg_bleu, summarization_score, grammar_score)
    best_rouge_scores.append(best_scores[0])
    best_bert_scores.append(best_scores[1])
    best_bleu_scores.append(best_scores[2])
    best_summarization_scores.append(best_scores[3])
    best_grammar_scores.append(best_scores[4])
    
    tqdm.write("Best Candidate this iteration: " + result_candidate)
    tqdm.write("Best Candidate Objectives (Summarization, Grammar): " + str(result_objectives))
    tqdm.write(f"Best Candidate Scores (ROUGE, BERT, BLEU, Summarization, Grammar): {best_scores}")
    if 'wandb' in globals():
        wandb.log({
            "Best Candidate": result_candidate,
            "Best Candidate Objectives": result_objectives,
            "Best ROUGE Score": best_scores[0],
            "Best BERT Score": best_scores[1],
            "Best BLEU Score": best_scores[2],
            "Best Summarization Score": best_scores[3],
            "Best Grammar Score": best_scores[4]
        })
    
    # Update patience logic
    if not hasattr(plot_pareto_front, 'best_candidate'):
        plot_pareto_front.best_candidate = result_candidate
        plot_pareto_front.patience_counter = 0
    else:
        if result_candidate == plot_pareto_front.best_candidate:
            plot_pareto_front.patience_counter += 1
        else:
            plot_pareto_front.best_candidate = result_candidate
            plot_pareto_front.patience_counter = 0
    
    if plot_pareto_front.patience_counter >= args.patience:
        tqdm.write("Ran out of patience")
        meta_file.write("Ran out of patience\n")
        break
    elif complete_phi4.count >= args.budget:
        tqdm.write("Ran out of budget")
        break
    
    torch.cuda.empty_cache()
    gc.collect()

plot_pareto_front(population_data, fronts, iter_idx, args.meta_dir, final=True)

if 'wandb' in globals():
    wandb.log({"Final Best Candidate": result_candidate, "Final Objectives": result_objectives})
meta_file.write('\n')
tqdm.write("APICalls for search: " + str(complete_phi4.count))
meta_file.write("APICalls: " + str(complete_phi4.count) + '\n')
if 'wandb' in globals():
    wandb.log({"APICalls": complete_phi4.count})

def plot_best_candidate_scores(meta_dir, rouge_scores, bert_scores, bleu_scores, summarization_scores, grammar_scores):
    # Iteration numbers start at 0 (best initial candidate)
    iterations = list(range(len(rouge_scores)))
    
    # Plot 1: Iteration vs ROUGE Score
    plt.figure(figsize=(8, 6))
    plt.plot(iterations, rouge_scores, marker='o', linestyle='-')
    plt.xlabel('Iteration Number')
    plt.ylabel('ROUGE Score')
    plt.title('Best Candidate ROUGE Score vs Iteration (0 = Best Initial Candidate)')
    plt.grid(True)
    plt.xticks(iterations)  # Ensure all iterations are shown
    plot_path = os.path.join(meta_dir, 'best_rouge_scores.png')
    plt.savefig(plot_path)
    plt.close()
    if 'wandb' in globals():
        wandb.log({"Best ROUGE Scores": wandb.Image(plot_path)})
    
    # Plot 2: Iteration vs BERT Score
    plt.figure(figsize=(8, 6))
    plt.plot(iterations, bert_scores, marker='o', linestyle='-')
    plt.xlabel('Iteration Number')
    plt.ylabel('BERT Score')
    plt.title('Best Candidate BERT Score vs Iteration (0 = Best Initial Candidate)')
    plt.grid(True)
    plt.xticks(iterations)
    plot_path = os.path.join(meta_dir, 'best_bert_scores.png')
    plt.savefig(plot_path)
    plt.close()
    if 'wandb' in globals():
        wandb.log({"Best BERT Scores": wandb.Image(plot_path)})
    
    # Plot 3: Iteration vs BLEU Score
    plt.figure(figsize=(8, 6))
    plt.plot(iterations, bleu_scores, marker='o', linestyle='-')
    plt.xlabel('Iteration Number')
    plt.ylabel('BLEU Score')
    plt.title('Best Candidate BLEU Score vs Iteration (0 = Best Initial Candidate)')
    plt.grid(True)
    plt.xticks(iterations)
    plot_path = os.path.join(meta_dir, 'best_bleu_scores.png')
    plt.savefig(plot_path)
    plt.close()
    if 'wandb' in globals():
        wandb.log({"Best BLEU Scores": wandb.Image(plot_path)})
    
    # Plot 4: Iteration vs Summarization Score
    plt.figure(figsize=(8, 6))
    plt.plot(iterations, summarization_scores, marker='o', linestyle='-')
    plt.xlabel('Iteration Number')
    plt.ylabel('Summarization Score')
    plt.title('Best Candidate Summarization Score vs Iteration (0 = Best Initial Candidate)')
    plt.grid(True)
    plt.xticks(iterations)
    plot_path = os.path.join(meta_dir, 'best_summarization_scores.png')
    plt.savefig(plot_path)
    plt.close()
    if 'wandb' in globals():
        wandb.log({"Best Summarization Scores": wandb.Image(plot_path)})
    
    # Plot 5: Iteration vs Grammar Score
    plt.figure(figsize=(8, 6))
    plt.plot(iterations, grammar_scores, marker='o', linestyle='-')
    plt.xlabel('Iteration Number')
    plt.ylabel('Grammar Score')
    plt.title('Best Candidate Grammar Score vs Iteration (0 = Best Initial Candidate)')
    plt.grid(True)
    plt.xticks(iterations)
    plot_path = os.path.join(meta_dir, 'best_grammar_scores.png')
    plt.savefig(plot_path)
    plt.close()
    if 'wandb' in globals():
        wandb.log({"Best Grammar Scores": wandb.Image(plot_path)})

# Plot the best candidate scores
plot_best_candidate_scores(
    args.meta_dir,
    best_rouge_scores,
    best_bert_scores,
    best_bleu_scores,
    best_summarization_scores,
    best_grammar_scores
)

tqdm.write("\nTesting ....")
meta_file.write("Testing ....\n")
final_score = score(result_candidate, 'test', write=args.write_preds)
tqdm.write("Final Candidate Test Score: " + str(final_score))
if 'wandb' in globals():
    wandb.log({"Final Candidate Test Score": final_score})
meta_file.write("Final Candidate Test Score: " + str(final_score) + '\n')
tqdm.write("Final Best Candidate: " + result_candidate)
meta_file.write("Final Best Candidate: " + result_candidate + '\n')
tqdm.write("APICalls: " + str(complete_phi4.count))
meta_file.write("APICalls: " + str(complete_phi4.count) + '\n')
end_time = time.time()
total_time = end_time - start_time
tqdm.write("Total execution time: {:.2f} seconds".format(total_time))
meta_file.write("Total execution time: {:.2f} seconds".format(total_time) + '\n')
if 'wandb' in globals():
    wandb.log({"Total Execution Time": total_time})

if 'wandb' in globals():
    wandb.save(meta_path)
meta_file.close()

global_progress_bar.close()

API Calls:   0%|          | 0/7000 [00:00<?, ?it/s]wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: urdusummarisation (urdusummarisation-indian-institute-of-information-techno) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


API Calls:   0%|          | 0/7000 [00:15<?, ?it/s]

Wandb is setup



config.json:   0%|          | 0.00/825 [00:00<?, ?B/s]

2025-05-13 06:29:00.438272: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747117740.859364      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747117740.986182      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


...

APICalls for search: 7448


API Calls: 7449it [6:41:14,  1.14it/s]


Testing ....


API Calls: 7548it [6:45:55,  3.16s/it]wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
                                      

Final Candidate Test Score: 46.316062150210186
Final Best Candidate: Write a brief sentence that encapsulates the primary topic of the provided text.
APICalls: 7548
Total execution time: 15939.74 seconds
